# 国债ETF 3秒快照数据检查与时间范围汇总

本 Notebook 完成以下任务：
1. 分别读取两个目录下全部 CSV 文件；
2. 输出每个目录的文件名列表；
3. 检查每个 CSV 的列是否与预期一致（缺失列 / 冗余列）；
4. 统计每个 CSV 文件 `trade_time` 的最小值与最大值；
5. 以适合后续合并与清洗的数据结构保存结果。

In [ ]:
from pathlib import Path
import pandas as pd

# 为了在 Notebook 中完整展示 DataFrame 列
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

In [ ]:
# ===== 1) 基础配置：目录与预期列 =====
base_dirs = {
    '511090': Path('国债ETF数据/3秒快照/511090'),
    '511130': Path('国债ETF数据/3秒快照/511130'),
}

expected_columns = [
    'Unnamed: 0', 'code', 'trade_time', 'pre_close', 'last', 'open', 'high', 'low', 'close',
    'volume', 'amount', 'num_trades', 'high_limited', 'low_limited',
    'ask_price1', 'ask_volume1', 'bid_price1', 'bid_volume1',
    'ask_price2', 'ask_volume2', 'bid_price2', 'bid_volume2',
    'ask_price3', 'ask_volume3', 'bid_price3', 'bid_volume3',
    'ask_price4', 'ask_volume4', 'bid_price4', 'bid_volume4',
    'ask_price5', 'ask_volume5', 'bid_price5', 'bid_volume5',
    'iopv', 'trading_phase_code'
]

# 收集每个目录下的 csv 文件（排序后更稳定，便于复现）
csv_files_by_dir = {
    k: sorted(v.glob('*.csv'))
    for k, v in base_dirs.items()
}

csv_files_by_dir

In [ ]:
# ===== 2) 一个 cell 输出每个文件夹所有 CSV 文件名 =====
for folder_key, files in csv_files_by_dir.items():
    print(f'\n目录 {folder_key} - 文件数量: {len(files)}')
    for f in files:
        print('  -', f.name)

In [ ]:
# ===== 3) 检查每个 CSV 的列完整性，并汇总 trade_time 范围 =====
# 使用 dict + DataFrame 的结构，便于后续做 merge / 清洗：
# 1) report_df: 文件级元信息（列检查 + 时间范围）
# 2) dataframes_by_dir: 真正的数据表对象（后续合并可直接用）

records = []
dataframes_by_dir = {k: {} for k in base_dirs.keys()}

for folder_key, files in csv_files_by_dir.items():
    for file_path in files:
        # 读取 CSV
        df = pd.read_csv(file_path)
        dataframes_by_dir[folder_key][file_path.name] = df

        actual_cols = df.columns.tolist()
        missing_cols = [c for c in expected_columns if c not in actual_cols]
        extra_cols = [c for c in actual_cols if c not in expected_columns]

        # trade_time 可能是字符串或数值，这里统一转成 datetime 尝试解析
        # errors='coerce' 遇到异常值会变成 NaT，后续清洗时可重点处理
        tt = pd.to_datetime(df['trade_time'], errors='coerce') if 'trade_time' in df.columns else pd.Series(dtype='datetime64[ns]')

        records.append({
            'folder': folder_key,
            'file_name': file_path.name,
            'row_count': len(df),
            'column_count': len(actual_cols),
            'is_column_match': (len(missing_cols) == 0 and len(extra_cols) == 0),
            'missing_columns': missing_cols,
            'extra_columns': extra_cols,
            'trade_time_min': tt.min(),
            'trade_time_max': tt.max(),
            'trade_time_nat_count': int(tt.isna().sum()) if len(tt) > 0 else None,
        })

report_df = pd.DataFrame(records).sort_values(['folder', 'file_name']).reset_index(drop=True)
report_df

In [ ]:
# ===== 4) 分目录查看列检查结果（缺失/冗余） =====
for folder_key in base_dirs.keys():
    print(f'\n===== 目录 {folder_key} 列检查结果 =====')
    sub = report_df.loc[report_df['folder'] == folder_key, ['file_name', 'is_column_match', 'missing_columns', 'extra_columns']]
    display(sub)

In [ ]:
# ===== 5) 分目录查看每个文件 trade_time 最小值与最大值 =====
for folder_key in base_dirs.keys():
    print(f'\n===== 目录 {folder_key} trade_time 范围 =====')
    sub = report_df.loc[
        report_df['folder'] == folder_key,
        ['file_name', 'trade_time_min', 'trade_time_max', 'trade_time_nat_count', 'row_count']
    ]
    display(sub)

In [ ]:
# ===== 6) （可选）后续合并前的建议数据结构说明 =====
# report_df: 文件级质量与时间范围索引表，可作为“数据目录表”
# dataframes_by_dir: {目录: {文件名: DataFrame}}，便于逐文件清洗后再 concat

print('report_df shape:', report_df.shape)
print('dataframes_by_dir keys:', list(dataframes_by_dir.keys()))
print('511090 文件数:', len(dataframes_by_dir['511090']))
print('511130 文件数:', len(dataframes_by_dir['511130']))